# Shrink a bloated system prompt with Alexandria

You maintain the system prompt for **TripGenie**, a travel-planning chatbot. Over months, every time
something went wrong you bolted on "just one more line." Now each section is a pile of instructions
that say almost the same thing three different ways — the prompt is long, repetitive, and you pay for
every one of those tokens on every single call.

**[Alexandria](https://github.com/ucsc-cse115a-alexandria/alexandria)** shrinks instruction-heavy
prompts *without labels, training, or a target answer*. It finds near-duplicate sentences with
embeddings and has an LLM merge each pair into one tighter sentence that keeps the meaning. Markdown
headings are protected, so it compacts your prompt **section by section**.

This notebook runs the whole loop with the `alexandria` **CLI**:

1. Look at the bloated prompt.
2. `alexandria reduce` it — one command.
3. See exactly what merged (the **diff**).
4. **Score** it — tokens saved vs. meaning kept.
5. Prove the shorter prompt still drives the **same chatbot behavior**.

> **Requirements:** Python 3.14, [uv](https://docs.astral.sh/uv/), and an OpenAI key. This notebook
> makes real OpenAI calls (embeddings for the reduction + a few `gpt-5.6-terra` chat completions);
> total cost is a fraction of a cent. Provide the key via a `.env` file (`OPENAI_API_KEY=...`) or an
> exported environment variable.

## Setup

In [1]:
import json
import subprocess
import tempfile
import textwrap
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                    # read OPENAI_API_KEY from a .env in this folder or a parent
openai_client = OpenAI()         # fails fast if no key is resolvable

CHAT_MODEL = "gpt-5.6-terra"     # the travel bot and the judge below both use this model
REASONING_EFFORT = "medium"      # gpt-5.6-terra is a reasoning model; try low / medium / high
PROMPT_FILE = Path("travel_agent_prompt.md")


def run_alexandria(*args: str) -> str:
    # Run the alexandria CLI and return its stdout (raises if the command exits non-zero).
    result = subprocess.run(
        ["uv", "run", "alexandria", *args],
        capture_output=True, text=True, check=True,
    )
    return result.stdout

## Step 1 — The prompt you're maintaining

Here's the current `TripGenie` system prompt. Four sections, each grown to **six bullets** that overlap
heavily. Read any section closely and you'll notice the lines pair up: two say the same thing about
tone, two about accuracy, and so on — just worded differently enough that nobody noticed the drift.

In [2]:
original_prompt = PROMPT_FILE.read_text()
print(original_prompt)

# Role

- TripGenie is a friendly travel-planning assistant that helps travelers plan their trips.
- Named TripGenie, you help travelers plan their trips as a friendly travel-planning assistant.
- You build a detailed day-by-day itinerary that covers the whole trip.
- For the whole trip, you build a detailed day-by-day itinerary.
- On booking flights, hotels, and transportation, you provide practical guidance.
- Practical guidance is what you provide for booking flights, hotels, and transportation.

# Communication

- Greet every traveler with genuine warmth and a friendly, welcoming tone.
- Every traveler deserves a warm, friendly, welcoming greeting and a genuine tone from you.
- Stay patient and empathetic with travelers who feel anxious or stressed about their trip.
- Travelers who feel anxious or stressed about their trip need you to be patient and empathetic.
- Keep a professional, respectful tone and never be rude, dismissive, or condescending to a traveler.
- A professional, re

## Step 2 — Reduce it with one command

```bash
alexandria reduce travel_agent_prompt.md          # reduced prompt on stdout
alexandria reduce travel_agent_prompt.md --json   # machine-readable summary
```

We use `--json` so we can inspect *what* changed. The LLM rewrites are stochastic, so exact wording
varies run to run — but which near-duplicate pairs get merged, and the overall reduction, are stable.

In [3]:
reduction = json.loads(run_alexandria("reduce", str(PROMPT_FILE), "--json"))
reduced_prompt = reduction["text"]

tokens_saved = reduction["source_tokens"] - reduction["reduced_tokens"]
print(f"{reduction['source_tokens']} → {reduction['reduced_tokens']} tokens "
      f"(−{tokens_saved}, {reduction['reduction_pct']:.0%} smaller)\n")
print(reduced_prompt)

460 → 205 tokens (−255, 55% smaller)

# Role

TripGenie is a friendly travel-planning assistant that helps travelers plan trips.
Build a detailed day-by-day itinerary for the whole trip.
Provide practical guidance for booking flights, hotels, and transportation.

# Communication

Greet every traveler genuinely, warmly, and with a friendly, welcoming tone.
Be patient and empathetic with travelers who feel anxious or stressed about their trip.
Maintain a professional, respectful tone with every traveler; never be rude, dismissive, or condescending.

# Accuracy

Never state prices, schedules, or availability unless they were actually provided.
When unsure of an answer, tell the traveler rather than guessing.
Before booking, travelers must confirm prices and times on the official airline or hotel site.

# Output Format

Present every trip plan as a day-by-day itinerary with a separate itinerary block for each day.
Keep every reply concise and skimmable; avoid dense walls of text.
Always di

## Step 3 — See what merged (the diff)

Each section keeps its heading and collapses from **6 near-duplicate bullets to ~3 tight sentences**.
For every merge, Alexandria reports the cosine similarity of the pair it fused — higher means the two
lines were closer in meaning, even when they read very differently.

In [4]:
def split_into_sections(prompt: str) -> dict[str, list[str]]:
    # Map each "# Heading" to the list of instruction lines beneath it.
    sections: dict[str, list[str]] = {}
    heading = None
    for line in prompt.splitlines():
        if line.startswith("# "):
            heading = line.removeprefix("# ").strip()
            sections[heading] = []
        elif line.strip() and heading is not None:
            sections[heading].append(line.strip().removeprefix("- "))
    return sections


before = split_into_sections(original_prompt)
after = split_into_sections(reduced_prompt)

for heading, original_lines in before.items():
    reduced_lines = after[heading]
    print(f"# {heading}   ({len(original_lines)} lines → {len(reduced_lines)})")
    for line in original_lines:
        print(f"  before   {line}")
    for line in reduced_lines:
        print(f"  after    {line}")
    print()

print("What Alexandria merged (cosine = how near-duplicate the fused pair was):")
for edit in reduction["applied"]:
    print(f"  • {edit['reason']}")

# Role   (6 lines → 3)
  before   TripGenie is a friendly travel-planning assistant that helps travelers plan their trips.
  before   Named TripGenie, you help travelers plan their trips as a friendly travel-planning assistant.
  before   You build a detailed day-by-day itinerary that covers the whole trip.
  before   For the whole trip, you build a detailed day-by-day itinerary.
  before   On booking flights, hotels, and transportation, you provide practical guidance.
  before   Practical guidance is what you provide for booking flights, hotels, and transportation.
  after    TripGenie is a friendly travel-planning assistant that helps travelers plan trips.
  after    Build a detailed day-by-day itinerary for the whole trip.
  after    Provide practical guidance for booking flights, hotels, and transportation.

# Communication   (6 lines → 3)
  before   Greet every traveler with genuine warmth and a friendly, welcoming tone.
  before   Every traveler deserves a warm, friendly, welcomi

## Step 4 — Score it: tokens saved vs. meaning kept

`alexandria compare` embeds both prompts and reports their cosine similarity alongside the token
reduction. `--min-similarity` turns that into an exit-code gate you can drop into CI to guarantee a
reduction never drifts too far from the original meaning.

In [5]:
reduced_file = Path(tempfile.mkdtemp()) / "reduced.md"
reduced_file.write_text(reduced_prompt)

score = json.loads(run_alexandria("compare", str(PROMPT_FILE), str(reduced_file)))
print(f"tokens:        {score['original_tokens']} → {score['edited_tokens']} "
      f"({score['token_reduction']:.0%} fewer)")
print(f"similarity:    {score['similarity']:.3f}   (1.000 = identical meaning)")
print(f"meaning drift: {score['cos_sim_diff']:.3f}   (1 − similarity)")

gate = subprocess.run(
    ["uv", "run", "alexandria", "compare", str(PROMPT_FILE), str(reduced_file),
     "--min-similarity", "0.9"],
    capture_output=True, text=True,
)
print(f"\nCI gate `--min-similarity 0.9`: {'passed' if gate.returncode == 0 else 'failed'}")

tokens:        460 → 205 (55% fewer)
similarity:    0.918   (1.000 = identical meaning)
meaning drift: 0.082   (1 − similarity)



CI gate `--min-similarity 0.9`: passed


## Step 5 — Does the shorter prompt still work?

Token counts are only half the story. The real question: does the compressed prompt still make the
chatbot **behave the same**? We wire both prompts into the same `gpt-5.6-terra` travel bot and ask
three questions, each probing a different section:

- **Plan a trip** → exercises *Role* + *Output Format* (day-by-day itinerary, prices with currency).
- **Ask for a live price** → exercises *Accuracy* (refuse to invent it, defer to official sources).
- **Vent about a cancelled flight** → exercises *Communication* (empathetic, practical).

In [6]:
def ask_tripgenie(system_prompt: str, question: str) -> str:
    # Answer one question as TripGenie, driven by the given system prompt.
    reply = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        reasoning_effort=REASONING_EFFORT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ],
    )
    return reply.choices[0].message.content.strip()


questions = [
    "I have 3 days in Kyoto in April. Can you plan my trip?",
    "How much is a flight from San Francisco to Tokyo next month?",
    "I'm really stressed, my flight got cancelled. What do I do?",
]

answers = []
for question in questions:
    answers.append({
        "question": question,
        "long_prompt": ask_tripgenie(original_prompt, question),
        "short_prompt": ask_tripgenie(reduced_prompt, question),
    })

for answer in answers:
    print("Q:", answer["question"])
    print("\n  long prompt →")
    print(textwrap.indent(answer["long_prompt"][:600], "    "))
    print("\n  short prompt →")
    print(textwrap.indent(answer["short_prompt"][:600], "    "))
    print("\n" + "─" * 80 + "\n")

Q: I have 3 days in Kyoto in April. Can you plan my trip?

  long prompt →
    Welcome to Kyoto! April is a wonderful time to visit—spring foliage and possible cherry blossoms make the city especially beautiful. Here’s a balanced 3-day itinerary that groups sights by area to minimize transit time.

    ## Day 1 — Higashiyama, Gion & Evening Streets
    **Morning — Southern Higashiyama**
    - Start early at **Fushimi Inari Taisha**. Walk through the iconic torii gates; allow 1.5–2.5 hours depending on how far up the mountain you go.
    - Take the Keihan Line or a taxi toward **Kiyomizu-dera**.

    **Midday — Historic lanes**
    - Visit **Kiyomizu-dera**, known for its wooden stage and views 

  short prompt →
    Welcome to Kyoto! April is a beautiful time to visit, with spring greenery and possible cherry blossoms depending on the exact dates. Expect popular areas to be busy, so start early each day and reserve key experiences in advance.

    ## Day 1 — Eastern Kyoto: Temples, Hig

### Judge it automatically

Eyeballing three answers is fine; let's also score it. We ask `gpt-5.6-terra` to judge **behavioral**
equivalence — whether both answers follow the same instructions — while explicitly ignoring length,
formatting, and wording.

In [7]:
JUDGE_INSTRUCTIONS = (
    "You compare two travel-assistant answers to the same question. One assistant used a long "
    "system prompt; the other used a compressed version of that same prompt. Judge BEHAVIORAL "
    "equivalence only -- do both follow the same instructions: give a day-by-day itinerary when "
    "asked to plan a trip, label prices with a currency, refuse to invent specific prices/times and "
    "defer to official sources, and stay empathetic and practical? IGNORE differences in length, "
    "formatting, and wording. Reply as JSON: "
    '{"equivalent": true/false, "reason": "<one sentence about the shared behavior>"}.'
)


def judge_equivalence(question: str, long_answer: str, short_answer: str) -> dict:
    # Ask the model whether the two answers follow the same instructions.
    comparison = (
        f"USER QUESTION:\n{question}\n\n"
        f"ANSWER A (long prompt):\n{long_answer}\n\n"
        f"ANSWER B (short prompt):\n{short_answer}"
    )
    reply = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        reasoning_effort=REASONING_EFFORT,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": JUDGE_INSTRUCTIONS},
            {"role": "user", "content": comparison},
        ],
    )
    return json.loads(reply.choices[0].message.content)


equivalent_count = 0
for answer in answers:
    verdict = judge_equivalence(answer["question"], answer["long_prompt"], answer["short_prompt"])
    equivalent_count += verdict["equivalent"]
    mark = "PASS" if verdict["equivalent"] else "FAIL"
    print(f"[{mark}] {answer['question']}")
    print(f"       {verdict['reason']}\n")

print(f"Behaviorally equivalent on {equivalent_count}/{len(answers)} questions.")

[PASS] I have 3 days in Kyoto in April. Can you plan my trip?
       Both provide practical, empathetic day-by-day Kyoto itineraries, avoid inventing prices or fixed operational details, and direct users to official sources or confirmation for variable information.



[PASS] How much is a flight from San Francisco to Tokyo next month?
       Both empathetically explain that they cannot provide a reliable live airfare price, direct the user to official airline/comparison sources, and request trip details to give practical follow-up help.



[PASS] I'm really stressed, my flight got cancelled. What do I do?
       Both respond empathetically with practical cancellation steps, avoid inventing specific prices or times, and defer rights and reimbursement details to airline policies and applicable official sources.

Behaviorally equivalent on 3/3 questions.


## Takeaways

- **Roughly half the tokens** (~55% fewer on this prompt), at **~0.92 cosine similarity** to the
  original meaning.
- The compressed prompt still produces a day-by-day itinerary, still refuses to invent prices, and
  still answers a stressed traveler with empathy — **behaviorally equivalent** on all three probes.
- The whole loop is one CLI command (`alexandria reduce`) plus a similarity gate you can put in CI.

Try it on your own prompt:

```bash
alexandria reduce your_prompt.md > reduced.md
alexandria compare your_prompt.md reduced.md --min-similarity 0.9
```

Prefer to approve edits yourself? Add `--interactive` (terminal) or `--browser` to review each
proposed merge and apply only the ones you accept.